[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/en_clase/S09_03_recomendadores_en_clase.ipynb)

# S09 · 03 — Recomendar experiencias a los huéspedes · versión en clase

**Tiempo estimado:** 50 min en clase · **Sprint 9** · M3 (UD 3.11, sistemas de recomendación)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/codigo_explicado/S09_03_recomendadores.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** Además del alojamiento, TurisData vende **25 experiencias** (excursiones,
talleres, deportes). Tenemos 2 232 valoraciones de 300 usuarios (de 1 a 5 estrellas). Cada usuario
solo ha valorado entre 4 y 11 experiencias: la mayoría de las casillas de la tabla están **vacías**.
Queremos decirle a cada cliente *"esto te puede gustar"*.

**Al terminar sabrás:**
1. Montar la matriz usuario × experiencia y separar entrenamiento y prueba.
2. Construir un recomendador por **popularidad** (la referencia que hay que batir).
3. Recomendar **por contenido** (categorías) y por **filtrado colaborativo** (gente parecida).
4. Entender la **factorización de matrices** como "gustos ocultos".
5. Evaluar con **RMSE** (error de la nota) y **precision@k** (¿acierta lo que recomienda?).

*No usamos ninguna librería especializada: todo con numpy, pandas y scikit-learn.*

> 🏫 **Versión «en clase»** · ⏱ ≈ 50 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2, 3**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_03_recomendadores.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

valoraciones = pd.read_csv(dato("valoraciones_experiencias.csv"))
experiencias = pd.read_csv(dato("experiencias.csv"))
print(valoraciones.shape, experiencias.shape)
valoraciones.head()

## 1. La matriz usuario × experiencia y la separación de prueba

**Idea.** Para saber si un recomendador es bueno hay que **esconderle** algunas valoraciones y ver si
las adivina. Reservamos un 20 % al azar como **prueba** (`prueba`) y construimos la matriz solo con el
80 % restante (`entrenamiento`). Las casillas vacías son `NaN`.

**Ejemplo resuelto:**

In [ ]:
prueba = valoraciones.sample(frac=0.2, random_state=SEMILLA)
entrenamiento = valoraciones.drop(prueba.index)

# filas = usuarios 1..300, columnas = experiencias 1..25
R = entrenamiento.pivot(index="id_usuario", columns="id_experiencia", values="valoracion")
R = R.reindex(index=range(1, 301), columns=range(1, 26))
M = R.notna().values                       # True donde hay valoración (matriz de "visto")
print("Matriz:", R.shape, "· casillas con dato:", f"{M.mean():.0%}")

media_global = entrenamiento["valoracion"].mean()
print("Nota media global:", round(media_global, 2))

# posiciones (fila, columna) de las valoraciones de prueba, para evaluar
fila_p = prueba["id_usuario"].values - 1
col_p = prueba["id_experiencia"].values - 1
real_p = prueba["valoracion"].values

def evaluar_rmse(P):
    """RMSE de una matriz de predicciones P (300 x 25) sobre las valoraciones de prueba."""
    pred = np.clip(P[fila_p, col_p], 1, 5)
    return float(np.sqrt(np.mean((pred - real_p) ** 2)))

**RMSE** = raíz del error cuadrático medio: *"de media, ¿en cuántas estrellas nos equivocamos?"*.
Cuanto más bajo, mejor. Punto de partida: predecir siempre la nota media global.

In [ ]:
P_global = np.full((300, 25), media_global)
print("RMSE prediciendo siempre la media global:", round(evaluar_rmse(P_global), 3))

## 2. Recomendador por popularidad

**Idea.** Recomendar a todos lo mejor valorado. Es simple, no personaliza y **casi siempre es la
referencia que hay que batir**. También es la única opción para un usuario nuevo (*arranque en frío*).

### Ejercicio 1 · 🏫 en clase
Calcula en `media_item` la nota media de cada experiencia **solo con la matriz de entrenamiento `R`**
(una `Series` de 25 valores). Después crea `P_pop`, una matriz de 300 × 25 en la que **todas las filas**
son `media_item` (`np.tile`), y guarda su error en `rmse_pop` con `evaluar_rmse`.

In [ ]:
# TODO: La media por columnas de un DataFrame promedia cada experiencia; np.tile sirve para repetir un vector en varias filas
...

assert len(media_item) == 25 and P_pop.shape == (300, 25), "media_item debe tener 25 valores y P_pop tamaño 300 x 25"
assert 0.95 < rmse_pop < 1.1, "El RMSE de popularidad debería rondar 1,0; revisa que uses R (solo entrenamiento)"
print(f"Ejercicio 1 correcto ✔ — RMSE popularidad = {rmse_pop:.3f}")

In [ ]:
top = media_item.sort_values(ascending=False).head(5)
print(experiencias.set_index("id_experiencia").loc[top.index, "nombre"].to_frame().assign(nota=top.round(2)))

## 3. Métrica de ranking: precision@k

**Idea.** El usuario no ve notas, ve **una lista corta**. Medimos si acertamos con ella:
**precision@k** = de las `k` experiencias recomendadas, ¿qué fracción le gustó de verdad (≥ 4 estrellas
en la prueba)? Se recomienda solo entre las que el usuario **no ha valorado** en entrenamiento.
Se promedia sobre los usuarios que tienen alguna experiencia "que le gusta" en la prueba.

Limitación: si recomendamos algo que le habría gustado pero nunca lo valoró, cuenta como fallo.
Por eso los valores son bajos y hay que **compararlos entre métodos**, no leerlos en absoluto.

### Ejercicio 2 · 🏫 en clase
Completa la función `precision_k(P, k=3)`. Para cada usuario de `relevantes`: (1) toma su fila de `P`,
(2) pon `-inf` en las experiencias que ya valoró en entrenamiento (fila de `M`), (3) quédate con las
`k` de mayor puntuación (`np.argsort`; suma 1 para pasar de posición a `id_experiencia`) y (4) calcula
qué fracción de esas `k` están en sus experiencias relevantes. Devuelve la media de todos los usuarios.

In [ ]:
relevantes = prueba[prueba["valoracion"] >= 4]      # lo que "le gusta" a cada usuario
print("Usuarios evaluables:", relevantes["id_usuario"].nunique())

def precision_k(P, k=3):
    aciertos = []
    # TODO: Recorre relevantes.groupby("id_usuario"); copia la fila P[u-1]; usa M[u-1] para ocultar lo ya visto; compara con set(...)
    ...
    return float(np.mean(aciertos))

p_pop = precision_k(P_pop)
azar = precision_k(np.random.default_rng(SEMILLA).random((300, 25)))
assert 0.09 < p_pop < 0.16, "precision@3 de popularidad debería rondar 0,12; revisa que ocultes lo ya valorado"
assert azar < p_pop + 0.02, "Recomendar al azar no debería superar claramente a popularidad; revisa la función"
print(f"Ejercicio 2 correcto ✔ — precision@3 popularidad = {p_pop:.3f} · azar = {azar:.3f}")

## 4. Recomendador basado en contenido

**Idea.** Si a un usuario le gustan las experiencias de *mar*, recomendarle más de *mar*. Cada
experiencia se describe por sus **características** (aquí, la categoría) y el usuario por **cuánto le
gustan** esas características, calculado con sus propias valoraciones.
Ventaja: funciona con experiencias **nuevas** que nadie ha valorado. Límite: solo ve lo que las
características describen.

Trabajamos con "residuos": lo que el usuario valora **por encima o por debajo** de la nota media de esa
experiencia (así separamos lo que es *bueno para todos* de lo que es *bueno para él*).

**Ejemplo resuelto:** matrices de características `F` y de residuos `C`.

In [ ]:
F = pd.get_dummies(experiencias.set_index("id_experiencia")["categoria"]).reindex(range(1, 26)).values.astype(float)
print("F (experiencia x categoría):", F.shape, "· categorías:", experiencias["categoria"].nunique())

media_item_v = media_item.fillna(media_global).values      # nota media de cada experiencia
C = np.where(M, R.values - media_item_v[None, :], 0.0)     # residuo (0 donde no hay valoración)

### Ejercicio 3 · 🏫 en clase
Calcula `perfil_usuario` (300 × 7): para cada usuario y categoría, la **suma de sus residuos en esa
categoría dividida entre (número de valoraciones en esa categoría + 3)**. El "+3" es prudencia: con pocas
valoraciones no nos fiamos. Pista: `C @ F` suma residuos por categoría y `M @ F` cuenta valoraciones
(convierte `M` a `float`).

In [ ]:
# TODO: Numerador y denominador son dos productos de matrices con F; el "+3" va solo en el denominador
...

assert perfil_usuario.shape == (300, 7), "perfil_usuario debe ser 300 usuarios x 7 categorías"
assert np.abs(perfil_usuario).max() < 3, "Los valores deben ser residuos pequeños (entre -3 y 3); revisa la división"
assert np.allclose(perfil_usuario[M.sum(1) == 0], 0), "Sin valoraciones el perfil debe ser 0"
print("Ejercicio 3 correcto ✔")

In [ ]:
# Predicción por contenido: nota media de la experiencia + afinidad del usuario con su categoría
P_cont = media_item_v[None, :] + perfil_usuario @ F.T
print("RMSE contenido:", round(evaluar_rmse(P_cont), 3), "· precision@3:", round(precision_k(P_cont), 3))

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_03_recomendadores.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
